# Backtest Sonuçları Analizi

Bu notebook, backtest_strategy.py ile oluşturulan backtest sonuçlarını analiz eder.

## İçerik:
1. Backtest sonuçlarını yükleme
2. Performans metrikleri
3. Equity curve analizi
4. Trade analizi (kazanan/kaybeden)
5. Drawdown analizi
6. Risk/Reward metrikleri
7. Sinyal analizi
8. Optimizasyon önerileri

**Yazar:** Trading Bot Sistemi  
**Faz:** 5  
**Tarih:** 2025-11-12

In [ ]:
# Gerekli kütüphaneler
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime

# Stil ayarları
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ Kütüphaneler yüklendi")

## 1. Backtest Sonuçlarını Yükleme

In [ ]:
# Backtest sonuç dosyasını yükle
results_file = '../backtest_results_report.json'

try:
    with open(results_file, 'r') as f:
        results = json.load(f)
    
    print("✅ Backtest sonuçları yüklendi")
    print(f"   Dosya: {results_file}")
    print(f"   Sembol: {results['symbol']}")
    print(f"   Toplam Trade: {results['total_trades']}")
    
except FileNotFoundError:
    print("❌ Backtest sonuç dosyası bulunamadı!")
    print("   Önce: python scripts/backtest_strategy.py --symbol BTCUSDT")
    results = None

## 2. Performans Metrikleri Özeti

In [ ]:
if results:
    print("=" * 60)
    print("📊 PERFORMANS METRİKLERİ")
    print("=" * 60)
    
    # Finansal metrikler
    print("\n💰 Finansal Metrikler:")
    print(f"   Başlangıç Bakiyesi: ${results['initial_balance']:,.2f}")
    print(f"   Final Bakiye: ${results['final_balance']:,.2f}")
    print(f"   Net Kar: ${results['net_profit']:,.2f} ({results['net_profit_percent']:.2f}%)")
    
    # Trade metrikleri
    print("\n📈 Trade Metrikleri:")
    print(f"   Toplam Trade: {results['total_trades']}")
    print(f"   Kazanan Trade: {results['winning_trades']} ({results['win_rate']:.2f}%)")
    print(f"   Kaybeden Trade: {results['losing_trades']}")
    print(f"   Profit Factor: {results['profit_factor']:.2f}")
    
    # Kazanç/Kayıp
    print("\n💵 Ortalama Kazanç/Kayıp:")
    print(f"   Ortalama Kazanç: ${results['avg_win']:,.2f}")
    print(f"   Ortalama Kayıp: ${results['avg_loss']:,.2f}")
    print(f"   Risk/Reward: {results['avg_win'] / results['avg_loss']:.2f}" if results['avg_loss'] > 0 else "   Risk/Reward: N/A")
    
    # Risk metrikleri
    print("\n⚠️ Risk Metrikleri:")
    print(f"   Max Drawdown: {results['max_drawdown']:.2f}%")
    
    print("=" * 60)

## 3. Equity Curve (Bakiye Eğrisi)

In [ ]:
if results and results.get('equity_curve'):
    # Equity curve DataFrame
    equity_df = pd.DataFrame(results['equity_curve'])
    
    # Görselleştir
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))
    
    # Equity curve
    ax1.plot(equity_df.index, equity_df['total_equity'], 
             linewidth=2, label='Total Equity', color='blue')
    ax1.axhline(y=results['initial_balance'], color='gray', 
               linestyle='--', alpha=0.5, label='Starting Balance')
    ax1.fill_between(equity_df.index, results['initial_balance'], 
                     equity_df['total_equity'], 
                     where=(equity_df['total_equity'] >= results['initial_balance']),
                     alpha=0.3, color='green', label='Profit')
    ax1.fill_between(equity_df.index, results['initial_balance'], 
                     equity_df['total_equity'],
                     where=(equity_df['total_equity'] < results['initial_balance']),
                     alpha=0.3, color='red', label='Loss')
    ax1.set_title('Equity Curve (Bakiye Eğrisi)', fontsize=14, fontweight='bold')
    ax1.set_ylabel('Bakiye (USDT)', fontsize=12)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Drawdown
    ax2.fill_between(equity_df.index, 0, -equity_df['drawdown'], 
                     alpha=0.6, color='red', label='Drawdown')
    ax2.set_title('Drawdown (%)', fontsize=14, fontweight='bold')
    ax2.set_ylabel('Drawdown (%)', fontsize=12)
    ax2.set_xlabel('Zaman', fontsize=12)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n📊 Equity Curve İstatistikleri:")
    print(f"   Peak Equity: ${equity_df['total_equity'].max():,.2f}")
    print(f"   Lowest Equity: ${equity_df['total_equity'].min():,.2f}")
    print(f"   Final Equity: ${equity_df['total_equity'].iloc[-1]:,.2f}")

## 4. Trade Analizi

In [ ]:
if results and results.get('trades'):
    # Trades DataFrame
    trades_df = pd.DataFrame(results['trades'])
    
    if len(trades_df) > 0:
        # PnL dağılımı
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        
        # 1. PnL histogram
        axes[0, 0].hist(trades_df['realized_pnl'], bins=30, 
                       color='steelblue', edgecolor='black', alpha=0.7)
        axes[0, 0].axvline(x=0, color='red', linestyle='--', linewidth=2)
        axes[0, 0].set_title('PnL Dağılımı', fontsize=12, fontweight='bold')
        axes[0, 0].set_xlabel('PnL (USDT)')
        axes[0, 0].set_ylabel('Trade Sayısı')
        axes[0, 0].grid(True, alpha=0.3)
        
        # 2. PnL zaman serisi
        cumulative_pnl = trades_df['realized_pnl'].cumsum()
        axes[0, 1].plot(cumulative_pnl.index, cumulative_pnl.values, 
                       linewidth=2, color='green')
        axes[0, 1].axhline(y=0, color='red', linestyle='--', alpha=0.5)
        axes[0, 1].set_title('Kümülatif PnL', fontsize=12, fontweight='bold')
        axes[0, 1].set_xlabel('Trade #')
        axes[0, 1].set_ylabel('Kümülatif PnL (USDT)')
        axes[0, 1].grid(True, alpha=0.3)
        
        # 3. Kazanan vs Kaybeden
        win_loss_data = [
            results['winning_trades'],
            results['losing_trades']
        ]
        axes[1, 0].pie(win_loss_data, labels=['Kazanan', 'Kaybeden'],
                      autopct='%1.1f%%', colors=['green', 'red'],
                      startangle=90)
        axes[1, 0].set_title('Kazanan vs Kaybeden Trade', 
                            fontsize=12, fontweight='bold')
        
        # 4. Trade süreleri (eğer varsa)
        if 'entry_time' in trades_df.columns and 'exit_time' in trades_df.columns:
            # TODO: Trade duration hesapla
            axes[1, 1].text(0.5, 0.5, 'Trade Duration\nAnalysis\n(TODO)', 
                          ha='center', va='center', fontsize=14)
        else:
            # Exit reason dağılımı
            if 'exit_reason' in trades_df.columns:
                exit_reasons = trades_df['exit_reason'].value_counts()
                axes[1, 1].bar(exit_reasons.index, exit_reasons.values,
                             color='coral', alpha=0.7)
                axes[1, 1].set_title('Exit Nedenleri', 
                                   fontsize=12, fontweight='bold')
                axes[1, 1].set_xlabel('Neden')
                axes[1, 1].set_ylabel('Sayı')
                axes[1, 1].tick_params(axis='x', rotation=45)
                axes[1, 1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # İstatistikler
        print(f"\n📊 Trade İstatistikleri:")
        print(f"   En Büyük Kazanç: ${trades_df['realized_pnl'].max():,.2f}")
        print(f"   En Büyük Kayıp: ${trades_df['realized_pnl'].min():,.2f}")
        print(f"   Ortalama PnL: ${trades_df['realized_pnl'].mean():,.2f}")
        print(f"   Medyan PnL: ${trades_df['realized_pnl'].median():,.2f}")

## 5. Sinyal Analizi

In [ ]:
if results and results.get('signals'):
    # Signals DataFrame
    signals_df = pd.DataFrame(results['signals'])
    
    if len(signals_df) > 0:
        print(f"\n📊 Sinyal İstatistikleri:")
        print(f"   Toplam Sinyal: {len(signals_df)}")
        print(f"   Toplam Trade: {results['total_trades']}")
        print(f"   Trade Conversion: {results['total_trades'] / len(signals_df) * 100:.1f}%")
        
        # Sinyal tipi dağılımı
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        
        # Sinyal tipleri
        signal_counts = signals_df['signal_type'].value_counts()
        ax1.bar(signal_counts.index, signal_counts.values, 
               color='steelblue', alpha=0.7)
        ax1.set_title('Sinyal Tipi Dağılımı', fontsize=12, fontweight='bold')
        ax1.set_xlabel('Sinyal Tipi')
        ax1.set_ylabel('Sayı')
        ax1.tick_params(axis='x', rotation=45)
        ax1.grid(True, alpha=0.3)
        
        # Güven skoru dağılımı
        ax2.hist(signals_df['confidence'], bins=20, 
                color='green', edgecolor='black', alpha=0.7)
        ax2.axvline(x=0.75, color='red', linestyle='--', 
                   linewidth=2, label='Min Threshold (0.75)')
        ax2.set_title('Sinyal Güven Skoru Dağılımı', 
                     fontsize=12, fontweight='bold')
        ax2.set_xlabel('Güven Skoru')
        ax2.set_ylabel('Sayı')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        print(f"\n   Ortalama Güven Skoru: {signals_df['confidence'].mean():.2%}")
        print(f"   Max Güven Skoru: {signals_df['confidence'].max():.2%}")
        print(f"   Min Güven Skoru: {signals_df['confidence'].min():.2%}")

## 6. Performance Karşılaştırması

In [ ]:
if results:
    # Performans metrikleri karşılaştırması
    metrics = {
        'Win Rate': results['win_rate'],
        'Profit Factor': results['profit_factor'] * 10,  # 0-100 skalası için
        'Net Profit %': results['net_profit_percent'],
        'Avg Win/Loss': (results['avg_win'] / results['avg_loss'] * 10) if results['avg_loss'] > 0 else 0
    }
    
    # Benchmark değerleri (iyi bir strateji için)
    benchmarks = {
        'Win Rate': 55,
        'Profit Factor': 15,  # 1.5 * 10
        'Net Profit %': 20,
        'Avg Win/Loss': 20  # 2.0 * 10
    }
    
    # Radar chart
    categories = list(metrics.keys())
    values = list(metrics.values())
    benchmark_values = list(benchmarks.values())
    
    # Açıları hesapla
    angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
    values += values[:1]
    benchmark_values += benchmark_values[:1]
    angles += angles[:1]
    
    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(projection='polar'))
    ax.plot(angles, values, 'o-', linewidth=2, label='Mevcut Strateji', color='blue')
    ax.fill(angles, values, alpha=0.25, color='blue')
    ax.plot(angles, benchmark_values, 'o-', linewidth=2, 
           label='Benchmark (İyi Strateji)', color='green', linestyle='--')
    ax.fill(angles, benchmark_values, alpha=0.15, color='green')
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories)
    ax.set_ylim(0, 100)
    ax.set_title('Performans Karşılaştırması', size=14, fontweight='bold', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
    ax.grid(True)
    
    plt.tight_layout()
    plt.show()

## 7. Optimizasyon Önerileri

In [ ]:
if results:
    print("\n" + "=" * 60)
    print("💡 OPTİMİZASYON ÖNERİLERİ")
    print("=" * 60)
    
    # Win rate analizi
    if results['win_rate'] < 50:
        print("\n⚠️ Kazanma Oranı Düşük (<50%):")
        print("   • Sinyal filtreleme kriterlerini artır")
        print("   • Minimum confidence threshold'u yükselt (>0.80)")
        print("   • İndikatör parametrelerini optimize et")
    
    # Profit factor analizi
    if results['profit_factor'] < 1.5:
        print("\n⚠️ Profit Factor Düşük (<1.5):")
        print("   • Risk/Reward oranını artır (hedef 2.0+)")
        print("   • Stop loss stratejisini gözden geçir")
        print("   • Trailing stop kullanmayı değerlendir")
    
    # Drawdown analizi
    if results['max_drawdown'] > 15:
        print("\n⚠️ Maksimum Drawdown Yüksek (>15%):")
        print("   • Pozisyon büyüklüğünü azalt")
        print("   • Max concurrent positions limitini düşür")
        print("   • Risk management kurallarını sıkılaştır")
    
    # Genel öneriler
    print("\n✅ Genel Öneriler:")
    print("   • Farklı market koşullarında test et (bull, bear, sideways)")
    print("   • Parametre optimizasyonu için grid search uygula")
    print("   • Walk-forward validation ile robustluğu test et")
    print("   • Gerçek market verisi ile forward test yap")
    
    print("=" * 60)

## Sonuç

Bu notebook'ta backtest sonuçları detaylı olarak analiz edildi:

- ✅ Performans metrikleri incelendi
- ✅ Equity curve ve drawdown analizi yapıldı
- ✅ Trade dağılımı görselleştirildi
- ✅ Sinyal kalitesi değerlendirildi
- ✅ Optimizasyon önerileri sunuldu

Sonraki adımlar:
1. Parametre optimizasyonu
2. Walk-forward validation
3. Paper trading testi
4. Live trading hazırlığı